# 19.5 倾向得分匹配 / Propensity Score Matching (PSM)

**中文**：19.4 讲了要控制混杂。但当混杂变量有很多个(年龄、学历、种族、收入历史……)时,怎么控制？直接按每个变量精确匹配"找一模一样的对照",维度一高就找不到了(维度诅咒)。**倾向得分(Propensity Score)** 给出天才的降维方案:把"这个人有多可能接受处理"压缩成**一个 0~1 的分数**,只按这一个分数匹配就够了。本节用因果推断史上最著名的数据集——**LaLonde 就业培训**——演示,并揭示一个惊人结论:**朴素的观测估计连正负号都是错的,而倾向得分方法能从观测数据里逼近随机实验的真值。**
**English**: 19.4 said to control confounders. But when there are many (age, education, race, income history…), how? Exactly matching on every variable to find "an identical control" fails as dimensions grow (the curse of dimensionality). The **propensity score** offers a brilliant dimension reduction: compress "how likely is this person to receive treatment" into **a single 0–1 score**, and match on that one score. We demonstrate on causal inference's most famous dataset — the **LaLonde job-training** study — and reveal a striking result: **the naive observational estimate gets even the sign wrong, while propensity methods recover the randomized-experiment truth from observational data.**

---

**中文**：设定:处理 $T\in\{0,1\}$(是否参加培训),结果 $Y$(之后的收入),协变量 $X$(混杂)。想要**处理的因果效应**。观测数据里,**接受处理的人和没接受的人本身就不一样**(自选择)——直接比会被混杂污染。
**English**: Setup: treatment $T\in\{0,1\}$ (attended training or not), outcome $Y$ (later earnings), covariates $X$ (confounders). We want the **causal effect of treatment**. In observational data, **the treated and untreated already differ** (self-selection) — a direct comparison is confounded.

**中文**：**倾向得分**定义为 $e(x)=P(T=1\mid X=x)$——给定协变量,一个人接受处理的概率(通常用逻辑回归估)。**Rosenbaum-Rubin 定理**是它的理论基石:**只要条件在倾向得分这一个标量上,就足以消除所有由 $X$ 引起的混杂**(倾向得分是"平衡得分")。于是高维匹配问题**降成一维**。
**English**: The **propensity score** is $e(x)=P(T=1\mid X=x)$ — the probability of being treated given covariates (usually estimated by logistic regression). The **Rosenbaum-Rubin theorem** is its foundation: **conditioning on the propensity score alone suffices to remove all confounding from $X$** (it is a "balancing score"). So the high-dimensional matching problem **collapses to one dimension**.

**中文**：拿到倾向得分后有几种用法:
**English**: Given propensity scores, several uses:
- **匹配(matching)**:给每个处理个体找一个倾向得分最接近的对照个体,配成对再比。
  **Matching**: for each treated unit, find the control with the closest propensity score, pair them, and compare.
- **逆倾向加权(IPW)**:给每个个体加权 $1/e$(处理组)或 $1/(1-e)$(对照组),制造一个"伪随机"的加权样本。
  **Inverse propensity weighting (IPW)**: weight each unit by $1/e$ (treated) or $1/(1-e)$ (control), creating a "pseudo-randomized" weighted sample.
- **分层(stratification)**:按倾向得分分箱,箱内比较再汇总。
  **Stratification**: bin by propensity score, compare within bins, then aggregate.

> 💡 **面试速查 / Interview cheat-sheet（★★★ 观测因果必考）**
> **中文**：**倾向得分 e(x)=P(T=1|X)** 把多维混杂压成一维(逻辑回归估), **Rosenbaum-Rubin**: 条件在 e(x) 上即可平衡 X(降维神器)。用法:**匹配 / IPW(逆倾向加权)/ 分层**。**两大关键假设**:①**无混杂(unconfoundedness/CIA)**:所有混杂都被 X 测到了——**这一条不可检验**, 是最大软肋(可能有未观测混杂);②**重叠/正性(overlap/positivity)**:每种人都有一定概率处理/不处理(0<e<1), 否则没法匹配。**必查**:匹配后**协变量平衡**(标准化差异<0.1, 画 love plot)。**双重稳健(AIPW/DR)**=倾向模型+结果模型, 只要有一个对就无偏。诚实:PSM 只能处理**观测到的**混杂; 未观测混杂要靠 IV/DiD/RDD。
> **English**: **Propensity score e(x)=P(T=1|X)** compresses multi-dim confounding to 1-D (estimated by logistic regression); **Rosenbaum-Rubin**: conditioning on e(x) balances X (a dimension-reduction gem). Uses: **matching / IPW / stratification**. **Two key assumptions**: ① **unconfoundedness (CIA)**: all confounders are measured in X — **untestable**, the biggest weakness (unobserved confounders may remain); ② **overlap/positivity**: every unit has some chance of treatment/control (0<e<1), else no match. **Always check**: **covariate balance** after matching (standardized difference <0.1, plot a love plot). **Doubly robust (AIPW/DR)** = propensity model + outcome model, unbiased if either is correct. Honest: PSM only handles **observed** confounders; unobserved confounding needs IV/DiD/RDD.


In [ ]:

# ============================================================
# 数据:LaLonde 就业培训(NSW 实验 + CPS 观测对照)/ LaLonde job training (NSW experiment + CPS controls)
# 中文:NSW 是随机实验(处理185人 vs 实验对照260人)→ 真实效应可算(黄金标准)。
#      但我们【假装】拿不到实验对照, 只能用一个非随机的观测对照(CPS人口普查, 15992人)——模拟真实观测研究的困境。
# English: NSW is an RCT (185 treated vs 260 experimental controls) → true effect is known (gold standard).
#      But we PRETEND we can't access the experimental controls and only have a non-random observational control
#      (CPS census, 15992 people) — mimicking a real observational study.
# ============================================================
import os, numpy as np, matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from scipy.spatial import cKDTree
R=os.path.expanduser("~/.cache/dsfs_causal")
cols=["treat","age","educ","black","hisp","married","nodegree","re74","re75","re78"]
treated=np.loadtxt(os.path.join(R,"nsw_treated.txt"))        # 处理组(参加培训)/ treated
exp_ctrl=np.loadtxt(os.path.join(R,"nsw_control.txt"))       # 实验对照(随机)/ experimental control
cps=np.loadtxt(os.path.join(R,"cps_control.txt"))            # 观测对照(非随机)/ observational control

exp_ate = treated[:,-1].mean() - exp_ctrl[:,-1].mean()      # 实验真值(黄金标准)/ RCT truth
naive   = treated[:,-1].mean() - cps[:,-1].mean()           # 朴素观测差 / naive observational diff
print(f"【黄金标准】随机实验估计的因果效应 / RCT ATE: ${exp_ate:.0f}  (培训确实提高了收入)")
print(f"【朴素观测】处理组 - CPS对照 / naive: ${naive:.0f}  (连正负号都反了!)")
print(f"混杂证据:处理组1975收入均值 ${treated[:,8].mean():.0f} vs CPS ${cps[:,8].mean():.0f}")
print("→ CPS 人群本来就富裕得多, 直接比等于拿苹果比橘子 / CPS is far richer to begin with")


**中文**：朴素观测估计给出 **−$8000 多**——荒谬地暗示"培训让人变穷"。真相是:CPS 对照组本来就是收入高得多的普通人群,而 NSW 处理组是**弱势群体**(失业者、低学历)。这就是典型的混杂。**倾向得分匹配**的任务:从 CPS 里挑出**和处理组"背景相似"**的那批人来做对照。先估倾向得分。
**English**: The naive observational estimate gives **−$8000+** — absurdly implying "training makes people poorer." The truth: the CPS controls are already a much higher-income general population, while the NSW treated are a **disadvantaged group** (unemployed, low education). Classic confounding. **Propensity score matching**'s job: pick from CPS those with a **similar background** to the treated as controls. First estimate propensity scores.


In [ ]:

# ============================================================
# 估计倾向得分 + 检查重叠 / estimate propensity scores + check overlap
# ============================================================
data=np.vstack([treated,cps]); T=data[:,0]; Y=data[:,9]
Xcov=data[:,1:9]                                             # 8 个协变量 / 8 covariates
Xs=StandardScaler().fit_transform(np.c_[Xcov, Xcov[:,-1]**2, Xcov[:,-2]**2])  # 加收入平方项 / add income^2
ps=LogisticRegression(max_iter=2000,C=1.0).fit(Xs,T).predict_proba(Xs)[:,1]   # 倾向得分 e(x) / propensity
ps=np.clip(ps,1e-4,1-1e-4)
ps_t=ps[T==1]; ps_c=ps[T==0]
print(f"倾向得分范围 / propensity range: 处理组 [{ps_t.min():.3f},{ps_t.max():.3f}], 对照组 [{ps_c.min():.3f},{ps_c.max():.3f}]")
print(f"重叠区(overlap)存在吗: 对照组里 e>0.1 的有 {(ps_c>0.1).sum()} 人 → 有可比对照")


**中文**：现在做两种主流估计:**最近邻匹配**(每个处理个体找倾向得分最近的 CPS 对照)和 **IPW**(逆倾向加权)。看它们能否把 −$8000 的荒谬结果,纠正回接近实验真值 +$1794。
**English**: Now two mainstream estimators: **nearest-neighbor matching** (each treated unit finds its closest-propensity CPS control) and **IPW** (inverse propensity weighting). See if they correct the absurd −$8000 back toward the experimental truth +$1794.


In [ ]:

# ============================================================
# 最近邻匹配 + IPW 估计 ATT / nearest-neighbor matching + IPW
# ============================================================
logit=np.log(ps/(1-ps))                                     # 在 logit 尺度上匹配更好 / match on logit scale
ti=np.where(T==1)[0]; ci=np.where(T==0)[0]
tree=cKDTree(logit[ci][:,None]); dist,j=tree.query(logit[ti][:,None],k=1)   # 每个处理个体最近的对照 / NN
matched_ctrl=ci[j]
att_match=(Y[ti]-Y[matched_ctrl]).mean()                    # 匹配后的处理效应(ATT)/ matched ATT

w=T + (1-T)*ps/(1-ps)                                        # ATT 的 IPW 权重 / IPW weights (for ATT)
att_ipw=(np.sum(T*Y*w)/np.sum(T*w)) - (np.sum((1-T)*Y*w)/np.sum((1-T)*w))

print(f"{'方法/method':<28}{'估计的因果效应':>16}")
print(f"{'黄金标准(随机实验)':<28}{exp_ate:>15.0f}  ← 真值")
print(f"{'朴素观测差 naive':<28}{naive:>15.0f}  (符号都错)")
print(f"{'倾向匹配 PSM (NN)':<28}{att_match:>15.0f}  ← 接近真值!")
print(f"{'逆倾向加权 IPW':<28}{att_ipw:>15.0f}  ← 也在正确范围")


In [ ]:

# ============================================================
# 检查协变量平衡 + 可视化 / covariate balance + visualization
# ============================================================
# 标准化均值差(SMD): 匹配前 vs 匹配后 / standardized mean difference before/after matching
names=["age","educ","black","hisp","married","nodegree","re74","re75"]
def smd(xt,xc): return (xt.mean()-xc.mean())/np.sqrt((xt.var()+xc.var())/2+1e-9)
smd_before=[smd(Xcov[T==1,k], Xcov[T==0,k]) for k in range(8)]
smd_after =[smd(Xcov[ti,k],   Xcov[matched_ctrl,k]) for k in range(8)]

fig,ax=plt.subplots(1,3,figsize=(17,4.7))
# ① 倾向得分分布(重叠)/ propensity distributions
ax[0].hist(ps_t,bins=30,alpha=0.6,color="#C44E52",density=True,label="处理组 treated")
ax[0].hist(ps_c,bins=30,alpha=0.6,color="#4C72B0",density=True,label="CPS 对照")
ax[0].set_title("倾向得分分布:处理组集中在低分区 / propensity overlap"); ax[0].set_xlabel("e(x)=P(处理)"); ax[0].legend(fontsize=8); ax[0].set_yscale("log")
# ② 协变量平衡(love plot)/ love plot
yy=np.arange(8)
ax[1].scatter(np.abs(smd_before),yy,c="#C44E52",label="匹配前 before",s=50)
ax[1].scatter(np.abs(smd_after), yy,c="#55A868",label="匹配后 after",s=50)
ax[1].axvline(0.1,ls="--",color="gray",label="平衡阈值 0.1"); ax[1].set_yticks(yy); ax[1].set_yticklabels(names,fontsize=8)
ax[1].set_title("协变量平衡:匹配后差异变小 / covariate balance"); ax[1].set_xlabel("|标准化均值差 SMD|"); ax[1].legend(fontsize=8)
# ③ ATE 对比 / ATE comparison
methods=["朴素\nnaive","IPW","PSM\n匹配","实验\ntruth"]; vals=[naive,att_ipw,att_match,exp_ate]
colors=["#C44E52","#DD8452","#55A868","#4C72B0"]
ax[2].bar(methods,vals,color=colors); ax[2].axhline(exp_ate,ls="--",color="#4C72B0")
ax[2].axhline(0,color="k",lw=0.5)
for i,v in enumerate(vals): ax[2].text(i,v,f"${v:.0f}",ha="center",va="bottom" if v>=0 else "top",fontsize=8)
ax[2].set_title("估计对比:PSM/IPW 纠回真值 / estimates vs truth"); ax[2].set_ylabel("因果效应($)")
plt.tight_layout(); plt.savefig("/tmp/ci05_viz.png",dpi=80); plt.show()
print(f"匹配后最大 |SMD|: {max(np.abs(smd_after)):.2f} (匹配前 {max(np.abs(smd_before)):.2f}) —— 混杂被大幅平衡")


**中文**：诚实解读:
**English**: Honest takeaways:

**中文**：
1. **倾向得分把观测数据"救回来了"**:朴素比较给出 −$8498(符号都反),而倾向匹配 +$2285、IPW +$1119——都**接近随机实验的真值 +$1794**。这是 Dehejia-Wahba 的著名结果,说明:**只要混杂都被测量到了,倾向得分方法能从观测数据里逼近随机实验的因果效应**。关键机制在中图——匹配后所有协变量的标准化差异都被压到阈值附近,处理组和对照组"背景终于可比了"。
2. **PSM 不完美,估计有波动**:匹配 $2285、IPW $1119,都在真值 $1794 附近但不精确(且依赖倾向模型设定、匹配方式、是否加平方项)。这很诚实:观测因果**永远没有随机实验干净**,不同方法/设定会给出一个范围。实践中要做**敏感性分析**、报告区间、多种方法交叉验证。
3. **最致命的假设不可检验**:PSM 的全部有效性建立在**"无未观测混杂"**上——即你测到了所有影响"是否处理"和"结果"的变量。**这一条无法从数据验证**(数据里没有的东西你看不见)。LaLonde 之所以能成功,是因为它有**处理前两年的收入(re74/re75)** 这个极强的混杂控制。如果有个你没测到的混杂(如"求职动机"),PSM 会照样给你一个自信但错误的答案。**所以:能随机化就别用 PSM;用 PSM 就要极力论证无未观测混杂,并做敏感性分析。**

**English**:
1. **The propensity score "rescues" observational data**: the naive comparison gives −$8498 (wrong sign), while PSM gives +$2285 and IPW +$1119 — all **close to the RCT truth +$1794**. This is the famous Dehejia-Wahba result: **as long as confounders are all measured, propensity methods can approximate the randomized-experiment causal effect from observational data**. The mechanism is the middle plot — after matching, every covariate's standardized difference is squeezed near the threshold; treated and control are "finally comparable in background."
2. **PSM isn't perfect; estimates wobble**: matching $2285, IPW $1119, both near the truth $1794 but imprecise (and dependent on the propensity model spec, matching method, and whether you add squared terms). Honestly: observational causation is **never as clean as a randomized experiment**; different methods/specs give a range. In practice do a **sensitivity analysis**, report intervals, and cross-check multiple methods.
3. **The most critical assumption is untestable**: PSM's entire validity rests on **"no unobserved confounding"** — that you measured every variable affecting both "treatment" and "outcome." **This cannot be verified from data** (you can't see what's not in the data). LaLonde works because it has **pre-treatment earnings for two years (re74/re75)** — an extremely strong confounder control. If there were an unmeasured confounder (like "job-seeking motivation"), PSM would still hand you a confident but wrong answer. **So: if you can randomize, don't use PSM; if you use PSM, argue hard for no unobserved confounding and do a sensitivity analysis.**

> 💼 **实战视角 / Practical angle**
> **中文**:PSM 是观测因果的**主力工具**(政策评估、医疗、用户增长里"某功能对留存的影响"当没法随机化时)。落地要点:①**倾向模型**用逻辑回归/GBM 都行, 重点是**平衡**而非预测精度;②**必查匹配后协变量平衡**(love plot, SMD<0.1), 不平衡就调模型/加项;③**修剪(trimming)** 掉无重叠区(e→0 或 1 的极端个体);④优先用**双重稳健(AIPW)**——同时建倾向和结果模型, 更抗设定错误;⑤**做敏感性分析**(Rosenbaum bounds)量化"要多强的未观测混杂才能推翻结论"。面试金句:*"倾向得分 e(x)=P(T|X) 把多维混杂降到一维(Rosenbaum-Rubin), 用匹配/IPW 消除观测混杂; 但依赖'无未观测混杂'这个不可检验的假设, 必须查协变量平衡+敏感性分析——LaLonde 上它把 −8000 的荒谬结果纠回了实验真值 +1794。"*
> **English**: PSM is a **workhorse** of observational causal inference (policy evaluation, healthcare, "a feature's effect on retention" when randomization isn't possible). Deployment keys: ① the **propensity model** can be logistic/GBM — the goal is **balance**, not predictive accuracy; ② **always check post-matching covariate balance** (love plot, SMD<0.1); fix the model if imbalanced; ③ **trim** the non-overlap region (extreme e→0 or 1); ④ prefer **doubly robust (AIPW)** — model both propensity and outcome, more robust to misspecification; ⑤ **do a sensitivity analysis** (Rosenbaum bounds) quantifying "how strong an unobserved confounder would need to be to overturn the conclusion." Interview line: *"Propensity score e(x)=P(T|X) reduces multi-dim confounding to 1-D (Rosenbaum-Rubin); matching/IPW removes observed confounding; but it relies on the untestable 'no unobserved confounding' assumption, so always check covariate balance + sensitivity — on LaLonde it corrected the absurd −8000 back to the experimental truth +1794."*

---
### 小结 / Summary
- **中文**:倾向得分 e(x)=P(T|X) 把多维混杂降一维(平衡得分); 用匹配/IPW/分层估因果效应。
- **English**: Propensity score e(x)=P(T|X) reduces multi-dim confounding to 1-D (a balancing score); estimate effects via matching/IPW/stratification.
- **中文**:LaLonde 上 PSM/IPW 把朴素的 −$8498 纠回接近实验真值 +$1794; 必查匹配后协变量平衡。
- **English**: On LaLonde, PSM/IPW correct the naive −$8498 back toward the experimental truth +$1794; always check post-matching balance.
- **中文**:致命假设=无未观测混杂(不可检验); 能随机化就别用观测法; 做敏感性分析。
- **English**: The fatal assumption = no unobserved confounding (untestable); if you can randomize, don't use observational methods; do sensitivity analysis.
